# Trajectory prediction training (scene-level + BiGRU)

Uses **multi-vehicle scene dataset** and **BiGRU temporal encoder**. Same setup as `explore_argoverse.ipynb`: clone repo, switch branch, download dataset.

## 1. Setup (Colab: clone repo, project path, install deps)

In [ ]:
# Clone project (Colab: run this first). If repo exists, clean cache and pull branch.
import os
import sys

REPO_URL = "https://github.com/PulockDas/Implement-STAST-System.git"  # set your repo URL
BRANCH = "master"  # branch to checkout
CLONE_DIR = "/content/Implement-STAST-System"  # Colab; use os.getcwd() for local

ip = get_ipython()
if os.path.isdir(CLONE_DIR) and os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    ip.run_line_magic("cd", CLONE_DIR)
    ip.system("git fetch origin")
    ip.system("git clean -fd")
    ip.system(f"git checkout {BRANCH}")
    ip.system(f"git pull origin {BRANCH}")
    print(f"Updated existing repo at {CLONE_DIR} (branch {BRANCH})")
else:
    parent = os.path.dirname(CLONE_DIR) or "."
    os.makedirs(parent, exist_ok=True)
    ip.run_line_magic("cd", parent)
    ip.system(f"git clone --branch {BRANCH} {REPO_URL} {os.path.basename(CLONE_DIR)}")
    print(f"Cloned repo to {CLONE_DIR} (branch {BRANCH})")

PROJECT_ROOT = CLONE_DIR
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
ip.run_line_magic("cd", PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
# If you didn't run the clone cell (e.g. local): set project root to parent of notebooks/.
import sys
import os

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
# Install dependencies (skip if already installed, e.g. in Colab)
!pip install -q tqdm

## 2. Download dataset with kagglehub

In [ ]:
import kagglehub

path = kagglehub.dataset_download("narendarmallireddy/argoverse1-motion-dataset")
print("Dataset path:", path)

In [ ]:
from pathlib import Path

data_path = Path(path)
csv_files = list(data_path.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s).")
if not csv_files:
    raise FileNotFoundError("No CSV files found under dataset path.")

## 3. Scene-level dataset and DataLoader

**How much data feeds the BiGRU:** We use **max_scenes** scenarios (one CSV = one scene = one training sample). Default below: **500** scenes. Each scene has multiple vehicles (past_traj shape N ≤ max_vehicles). So total training samples = number of valid scenes built (≤ max_scenes).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets.argoverse_dataset import ArgoverseSceneDataset
from utils.config import PAST_STEPS, FUTURE_STEPS

max_scenes = 500  # number of scenarios (CSVs) used for training
scene_csvs = csv_files[:max_scenes]
scene_dataset = ArgoverseSceneDataset(
    scene_csvs,
    past_steps=PAST_STEPS,
    future_steps=FUTURE_STEPS,
    max_vehicles=20,
    distance_threshold=50.0,
)
loader = DataLoader(scene_dataset, batch_size=32, shuffle=True, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Number of scene samples (training data size):", len(scene_dataset))
print("Device:", device)

## 4. BiGRU temporal encoder — shape test

In [ ]:
from models.temporal_encoder import BiGRUTrajectoryEncoder

batch = next(iter(loader))
past = batch["past_traj"]  # (B, N, 20, 2)

model = BiGRUTrajectoryEncoder()
vehicle_features = model.encoder(past)
print("vehicle_features.shape:", tuple(vehicle_features.shape))
print("expected: (B, N, 128)")

pred_traj = model(past)
print("pred_traj.shape:", tuple(pred_traj.shape))
print("expected: (B, 30, 2)")

## 5. Train BiGRU (MSE on future_target)

In [ ]:
from metrics.trajectory_metrics import ade, fde

model = BiGRUTrajectoryEncoder().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
EPOCHS = 10
for epoch in range(EPOCHS):
    total_loss = 0.0
    total_ade = 0.0
    total_fde = 0.0
    n_batches = 0
    for batch in loader:
        past_scene = batch["past_traj"].to(device)
        future_target = batch["future_target"].to(device)
        optimizer.zero_grad()
        pred = model(past_scene)
        loss = criterion(pred, future_target)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            total_loss += loss.item()
            total_ade += ade(pred, future_target).item()
            total_fde += fde(pred, future_target).item()
            n_batches += 1
    print(f"Epoch {epoch+1}/{EPOCHS}  MSE: {total_loss/max(n_batches,1):.6f}  ADE: {total_ade/max(n_batches,1):.4f}  FDE: {total_fde/max(n_batches,1):.4f}")
print("Done.")